In [4]:
#Import Libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns    
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from scikeras.wrappers import KerasRegressor
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor 
from catboost import CatBoostRegressor 
from lightgbm import LGBMRegressor 
import tensorflow as tf
from tensorflow import keras

DATA PREPROCESSING

In [5]:
# Importing the dataset
try:
    train = pd.read_csv('train.csv')
    test = pd.read_csv('test.csv')
except Exception as e:
    print(f'Error saat membaca file: {e}')
    print('Pastikan file train.csv dan test.csv ada di direktori yang sama dengan skrip ini.')

seed = 42

In [4]:
# INITIAL DATA PREPARATION

# Seperate target variable from features
y = train.price
X = train.drop(columns=['price'])

# Divide data now into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=seed)

# Select data with dtype int or float
numeric_features_train = X_train.select_dtypes(include=['int64', 'float64'])
numeric_features_test = X_val.select_dtypes(include=['int64', 'float64'])
low_missing_columns = numeric_features_train.columns[numeric_features_train.isnull().mean() < 0.5]

# Select categorical columns with relatively low cardinality (convenient but arbitrary)
low_cardinality_columns = [cname for cname in X_train.columns if X_train[cname].nunique() < 10 and X_train[cname].dtype == 'object']

In [3]:
sns.heatmap(X_train[low_missing_columns].isnull(), cbar=False, annot=True, cmap='viridis')

NameError: name 'sns' is not defined

In [ ]:
# NUMERICAL FEATURES HANDLING

# Select numeric features with less than 50% missing values
missing_threshold = 0.5
numeric_features_train = numeric_features_train[low_missing_columns]
numeric_features_test = numeric_features_test[low_missing_columns]

# Impute missing values
my_imputer = SimpleImputer(strategy='mean')
imputed_X_train_numeric = pd.DataFrame(my_imputer.fit_transform(numeric_features_train))
imputed_X_val_numeric = pd.DataFrame(my_imputer.transform(numeric_features_test))

# Imputation removed column names; put them back
imputed_X_train_numeric.columns = numeric_features_train.columns
imputed_X_val_numeric.columns = numeric_features_test.columns

# CATEGORICAL FEATURES HANDLING
# Make copy to avoid changing original data 
label_X_train = X_train.copy()
label_X_val = X_val.copy()

# Apply ordinal encoder to each column with categorical data
ordinal_encoder = OrdinalEncoder()
label_X_train[low_cardinality_columns] = ordinal_encoder.fit_transform(X_train[low_cardinality_columns])
label_X_val[low_cardinality_columns] = ordinal_encoder.fit_transform(X_val[low_cardinality_columns])
test[low_cardinality_columns]= ordinal_encoder.fit_transform(test[low_cardinality_columns])
X_train[low_cardinality_columns]

review_columns_remove = [
    'id', 
    'host_id', 
    'host_listings_count', 
    'number_of_reviews', 
    'number_of_reviews_l30d', 
    'availability_30',
    'availability_60',
    'availability_90',
]
final_X_train_numeric = imputed_X_train_numeric.drop(columns=review_columns_remove)
final_X_val_numeric = imputed_X_val_numeric.drop(columns=review_columns_remove)

final_X_train = final_X_train_numeric.join(label_X_train[low_cardinality_columns], how='left')
final_X_val = final_X_val_numeric.join(label_X_val[low_cardinality_columns], how='left')

print(final_X_train_numeric.columns.tolist())
print(final_X_val_numeric.columns.tolist())
final_X_train.drop()

['host_total_listings_count', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'availability_365', 'number_of_reviews_ltm', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'reviews_per_month']
['host_total_listings_count', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'availability_365', 'number_of_reviews_ltm', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'reviews_per_month']


MODEL XGBOOST REGRESSOR

In [ ]:
# Model Training XGBoost Regressor
xgb_model = XGBRegressor(
    n_estimators=200, 
    learning_rate=0.1,
    random_state=seed,  
    n_jobs=-1,
)

xgb_model.fit(final_X_train_numeric, y_train)
y_pred_train = xgb_model.predict(imputed_X_train_numeric)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))

print(f'RMSE on training set: {rmse_train:.2f}')

# Predict on validation set
y_pred_val = xgb_model.predict(imputed_X_val_numeric)
rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))

print(f'RMSE on validation set: {rmse_val:.2f}')

RMSE on training set: 107.47
RMSE on validation set: 110.92


In [14]:
submission_file = 'submission_4.csv'
test_ids = test['id']
price = xgb_model.predict(test[low_missing_columns])
submission = pd.DataFrame({'id': test_ids, 'price': price})
submission.to_csv(submission_file, index=False)

MODEL CATBOOST REGRESSOR

In [138]:
# Model Training CatBoost Regressor
cat_model = CatBoostRegressor(
    iterations=1300, 
    learning_rate=0.1,
    depth=10,
    random_state=seed,
    verbose=0
)

# Predict on training set with CatBoost
cat_model.fit(final_X_train, y_train)
y_pred_train_cat = cat_model.predict(final_X_train)
rmse_train_cat = np.sqrt(mean_squared_error(y_train, y_pred_train_cat))
print(f'RMSE on training set (CatBoost): {rmse_train_cat:.2f}')

# Predict on validation set with CatBoost
y_pred_val_cat = cat_model.predict(final_X_val)
rmse_val_cat = np.sqrt(mean_squared_error(y_val, y_pred_val_cat))
print(f'RMSE on validation set (CatBoost): {rmse_val_cat:.2f}')

RMSE on training set (CatBoost): 89.00
RMSE on validation set (CatBoost): 108.86


In [137]:
# Submission for CatBoost
submission_file_cat = 'submission_cat_5.csv'
price_cat = cat_model.predict(test[final_X_val.columns])
submission_cat = pd.DataFrame({'id': test_ids, 'price': price_cat})
submission_cat.to_csv(submission_file_cat, index=False)

MODEL LIGHTGBM REGRESSOR

In [147]:
# Model LightGBM Regressor
lgb_model = LGBMRegressor(
    n_estimators=2000, 
    learning_rate=0.1,
    num_leaves=70,
    random_state=seed,
    n_jobs=-1
)

# Predict on training set with LightGBM
lgb_model.fit(final_X_train, y_train)
y_pred_train = lgb_model.predict(final_X_train)
rmse_train_lgb = np.sqrt(mean_squared_error(y_train, y_pred_train))
print(f'RMSE on training set (LightGBM): {rmse_train_lgb:.2f}')

# Predict on validation set with LightGBM
y_pred_val_lgb = lgb_model.predict(final_X_val)
rmse_val_lgb = np.sqrt(mean_squared_error(y_val, y_pred_val_lgb))
print(f'RMSE on validation set (LightGBM): {rmse_val_lgb:.2f}')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007082 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2758
[LightGBM] [Info] Number of data points in the train set: 209515, number of used features: 24
[LightGBM] [Info] Start training from score 207.230566
RMSE on training set (LightGBM): 71.03
RMSE on validation set (LightGBM): 108.57


In [148]:
# Submission for LightGBM
submission_file_lgb = 'submission_lgb_3.csv'
price_lgb = lgb_model.predict(test[final_X_train.columns])
submission_lgb = pd.DataFrame({'id': test_ids, 'price': price_lgb})
submission_lgb.to_csv(submission_file_lgb, index=False)

MODEL STACKING

In [31]:
# Stacking Models
from sklearn.ensemble import StackingRegressor
# Define the base models
base_models = [
    ('xgb', XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=seed, n_jobs=-1)),
    ('cat', CatBoostRegressor(iterations=1400, learning_rate=0.15, depth=10, random_state=seed, verbose=0)),
    ('lgb', LGBMRegressor(n_estimators=1000, learning_rate=0.1, num_leaves=31, random_state=seed, n_jobs=-1))
]
# Define the meta-model
meta_model = LGBMRegressor(n_estimators=1000, learning_rate=0.1, num_leaves=31, random_state=seed, n_jobs=-1)
stacking_model = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    n_jobs=-1
)
# Fit the stacking model
stacking_model.fit(final_X_train_numeric, y_train)
# Predict on validation set with stacking model
y_pred_val_stacking = stacking_model.predict(final_X_val_numeric)
rmse_val_stacking = np.sqrt(mean_squared_error(y_val, y_pred_val_stacking))
print(f'RMSE on validation set (Stacking): {rmse_val_stacking:.2f}')


KeyboardInterrupt: 

In [ ]:
# Submission for Stacking
submission_file_stacking = 'submission_stacking_1.csv'
test_ids = test['id']
price_stacking = stacking_model.predict(test[final_X_train_numeric])
submission_stacking = pd.DataFrame({'id': test_ids, 'price': price_stacking})
submission_stacking.to_csv(submission_file_stacking, index=False)

c:\Users\aufar\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


MODEL RANDOM FOREST REGRESSOR

In [150]:
# Model Using Random Forest Regressor
from sklearn.ensemble import RandomForestRegressor
rf_model = RandomForestRegressor(
    n_estimators=1200,
    max_depth=50, 
    random_state=seed, 
    n_jobs=-1
)
rf_model.fit(final_X_train, y_train)
y_pred_train_rf = rf_model.predict(final_X_train)   
rmse_train_rf = np.sqrt(mean_squared_error(y_train, y_pred_train_rf))
print(f'RMSE on training set (Random Forest): {rmse_train_rf:.2f}') 

# Predict on validation set with Random Forest
y_pred_val_rf = rf_model.predict(final_X_val)
rmse_val_rf = np.sqrt(mean_squared_error(y_val, y_pred_val_rf))
print(f'RMSE on validation set (Random Forest): {rmse_val_rf:.2f}')

RMSE on training set (Random Forest): 41.83
RMSE on validation set (Random Forest): 110.47


In [152]:
submission_file_rf = 'submission_rf_5.csv'
test_ids = test['id']
price_rc = rf_model.predict(test[final_X_train.columns])
submission_df = pd.DataFrame({'id': test_ids, 'price': price_rc})
submission_df.to_csv(submission_file_rf, index=False)

MODEL USING DEEP LEARNING

In [77]:
# Pra-pemrosesan Data untuk Deep Learning

# Pipeline untuk pra-pemrosesan data numerik
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer()),
    ('scaler', StandardScaler())
])

# Pipeline untuk pra-pemrosesan data kategorikal
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Gabungkan Tranformer berbeda
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, low_missing_columns),
        ('cat', categorical_transformer, low_cardinality_columns)
    ],
    remainder='passthrough'  # Biarkan kolom lain tidak berubah
)

In [78]:
# Membangun model neural network
def create_nn_model(input_dim, learning_rate=0.001, hidden_units=(64,32), dropout_rate=0.0):
    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(input_dim,)))

    # Hidden layers
    for units in hidden_units:
        model.add(keras.layers.Dense(units, activation='relu', kernel_initializer='he_normal'))
        if dropout_rate > 0:
            model.add(keras.layers.Dropout(dropout_rate))
    # Output layer
    model.add(keras.layers.Dense(1, activation='linear'))

    # Optimizer
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    # Compile the model
    model.compile(optimizer=optimizer, loss='mean_squared_error', metrics=[tf.keras.metrics.RootMeanSquaredError()])
    return model

In [79]:
# --- 5. Buat Pipeline Utama dengan Scikeras KerasRegressor ---
# KerasRegressor memungkinkan model Keras diintegrasikan ke dalam pipeline scikit-learn.
nn_model = KerasRegressor(
    model=create_nn_model,
    # Parameter untuk fungsi create_nn_model:
    input_dim=None, # Scikeras akan mengisi ini secara otomatis setelah preprocessor di-fit
    learning_rate=0.001,
    hidden_units=(128, 64), # Contoh: dua lapisan tersembunyi dengan 128 dan 64 neuron
    dropout_rate=0.2, # Contoh: 20% dropout

    # Parameter untuk pelatihan Keras (metode fit):
    epochs=100,      # Jumlah iterasi pelatihan penuh pada seluruh dataset
    batch_size=32,   # Jumlah sampel per pembaruan bobot
    verbose=0,       # Matikan output log selama training (set to 1 for progress bar)
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', # Pantau loss pada data validasi
            patience=10,        # Berhenti jika loss tidak membaik selama 10 epoch
            restore_best_weights=True # Gunakan bobot terbaik yang ditemukan
        )
    ]
)

full_pipeline_nn = Pipeline(steps=[('preprocessor', preprocessor),
                                    ('regressor', nn_model)])

In [80]:
# Pelatihan model neural network
full_pipeline_nn.fit(X_train, y_train)

ValueError: Input contains NaN